In [ ]:

from dotenv import load_dotenv
from google.colab import drive
import os

drive.mount('/content/drive')
load_dotenv('/content/drive/MyDrive/llm-training-pipeline/.env')
os.chdir('/content/drive/MyDrive/llm-training-pipeline')

!git pull


In [ ]:

!pip install -q trl peft bitsandbytes transformers accelerate datasets wandb


In [ ]:

import gc
import json
import math
import os
import time

import torch
import wandb
from datasets import load_dataset
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import DPOConfig, DPOTrainer


In [ ]:

from huggingface_hub import login

login(token=os.getenv("HF_TOKEN"))
wandb.login(key=os.getenv("WANDB_API_KEY"))


In [ ]:

model_id = "Qwen/Qwen2.5-0.5B"
# Phase 2's best-generalizing run (eval 0.8903 vs the r=64 baseline's 0.9122), re-run with
# checkpointing on specifically so DPO could start from it.
SFT_ADAPTER = "checkpoints/sft-r8-n5000/final"
SFT_LORA_RANK = 8
MERGED_DIR = "checkpoints/sft-r8-merged"    # SFT weights baked into the base, see below

# Identical to the SFT template. DPO is a correction on top of a policy that was trained
# on this exact string -- change a newline here and the gradient spends itself fighting a
# formatting mismatch instead of shifting preferences.
PROMPT_TEMPLATE = "### Instruction:\n{instruction}\n\n### Response:\n"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

os.makedirs("results", exist_ok=True)
print(f"adapter exists: {os.path.isdir(SFT_ADAPTER)}")


In [ ]:

# CodeUltraFeedback: 9.5k code-instruction preference pairs, already binarized into
# chosen/rejected by GPT-4 ratings. Picked over generic UltraFeedback or Anthropic HH
# because Phase 2 fine-tuned on code and Phase 4 evaluates on code -- aligning on
# open-domain chat preferences would push the model off the domain it was trained for.
raw = load_dataset("coseal/CodeUltraFeedback_binarized", split="train")
print(raw)

ex = raw[0]
print("\n--- instruction ---")
print(ex["instruction"][:400])
print(f"\n--- chosen (rating {ex['rating_chosen']}, {ex['model_chosen']}) ---")
print(ex["chosen"][:400])
print(f"\n--- rejected (rating {ex['rating_rejected']}, {ex['model_rejected']}) ---")
print(ex["rejected"][:400])


In [ ]:

# Preference *strength* filter. Every pair carries 1-5 ratings for both sides; a 4-vs-3
# pair is a coin flip dressed up as a label, a 5-vs-1 pair is unambiguous. DPO's loss is
# a log-sigmoid of the reward difference, so noisy pairs do not average out -- they
# actively push the policy in the wrong direction. Trading pairs for label quality is the
# single biggest lever on whether DPO moves at all.
MIN_RATING_MARGIN = 2
N_TRAIN = 1000
N_VAL = 150
MAX_PROMPT_TOKENS = 256
MAX_COMPLETION_TOKENS = 384


def n_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def to_dpo_format(example):
    return {
        "prompt": PROMPT_TEMPLATE.format(instruction=example["instruction"]),
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }


strong = raw.filter(
    lambda e: (e["rating_chosen"] - e["rating_rejected"]) >= MIN_RATING_MARGIN
)
print(f"rating margin >= {MIN_RATING_MARGIN}: {len(strong)} / {len(raw)} pairs")

# Anything that would be truncated mid-response teaches the model to prefer a cut-off
# answer, so drop over-length pairs instead of letting the collator clip them.
fits = strong.filter(
    lambda e: n_tokens(PROMPT_TEMPLATE.format(instruction=e["instruction"])) <= MAX_PROMPT_TOKENS
    and n_tokens(e["chosen"]) <= MAX_COMPLETION_TOKENS
    and n_tokens(e["rejected"]) <= MAX_COMPLETION_TOKENS
)
print(f"also fits in {MAX_PROMPT_TOKENS}+{MAX_COMPLETION_TOKENS} tokens: {len(fits)} pairs")

pairs = fits.map(to_dpo_format, remove_columns=fits.column_names)
split = pairs.train_test_split(test_size=N_VAL, seed=42)
train_pairs = split["train"].select(range(min(N_TRAIN, len(split["train"]))))
val_pairs = split["test"]

print(f"\ntrain: {len(train_pairs)}  val: {len(val_pairs)}")
print("\n--- formatted prompt ---")
print(train_pairs[0]["prompt"])

# Length bias check on the labels themselves: if chosen is systematically longer than
# rejected, DPO will learn "longer = better" as a shortcut and any length inflation seen
# later is the dataset's fault, not the algorithm's. Worth knowing before training.
c = sum(len(e) for e in train_pairs["chosen"]) / len(train_pairs)
r = sum(len(e) for e in train_pairs["rejected"]) / len(train_pairs)
print(f"\nmean chars -- chosen {c:.0f}, rejected {r:.0f}  (ratio {c / r:.2f}x)")


In [ ]:

# Bake the SFT adapter into the base weights.
#
# Why not just keep training the existing adapter? Because of what serves as the DPO
# reference model. With PEFT, TRL computes reference log-probs by *disabling the adapter*
# rather than holding a second model in VRAM -- so with the SFT adapter attached, the
# reference would be the raw base model, and beta would be constraining drift from a
# model we already deliberately moved away from. Merging first makes adapter-disabled ==
# SFT, so the KL term measures drift from SFT, which is what beta is supposed to control.
#
# Caveat worth stating: the adapter was trained against 4-bit NF4 weights and is being
# merged into fp16 ones, so the deltas land on slightly different weights than they were
# fit to. Standard practice and the error is small at this scale, but it is not free.
base_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
merged = PeftModel.from_pretrained(base_fp16, SFT_ADAPTER).merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"merged SFT model written to {MERGED_DIR}")

del merged, base_fp16
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:

def run_dpo(run_name, beta, epochs=1, save_adapter=False):
    """One DPO run. Only beta varies between calls; everything else is held fixed.

    Returns a metrics dict and writes the full log history to results/.
    """
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    # Fresh 4-bit copy of the *merged* SFT model, so adapter-disabled == SFT reference.
    policy = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    policy = prepare_model_for_kbit_training(policy, use_gradient_checkpointing=True)

    # r=16, in the same low range as the SFT run this starts from. DPO is a preference
    # correction on an already-competent policy, not a domain transfer -- and the Phase 2
    # rank ablation showed r=64 had more capacity than the data could support and spent
    # the surplus memorizing.
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )

    total_steps = math.ceil(len(train_pairs) / 16) * epochs
    eval_steps = max(5, total_steps // 8)

    args = DPOConfig(
        output_dir=f"checkpoints/{run_name}",
        beta=beta,
        num_train_epochs=epochs,
        per_device_train_batch_size=1,     # 4 forward passes per example (chosen/rejected
        per_device_eval_batch_size=1,      # x policy/reference), so keep the batch tiny
        gradient_accumulation_steps=16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        learning_rate=5e-5,                # ~10x lower than SFT: small, targeted update
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        weight_decay=0.0,
        max_length=MAX_PROMPT_TOKENS + MAX_COMPLETION_TOKENS,
        max_prompt_length=MAX_PROMPT_TOKENS,
        eval_strategy="steps",
        eval_steps=eval_steps,
        save_strategy="no",
        logging_steps=max(5, eval_steps // 2),
        # Back to the standard fp16 + GradScaler path rather than SFT's force-everything-
        # to-fp16 workaround. DPO's loss is a log-sigmoid of small reward *differences*,
        # which is exactly where unscaled fp16 gradients underflow to zero.
        fp16=True,
        bf16=False,
        optim="adamw_torch",
        seed=42,
        report_to="wandb",
        run_name=run_name,
    )

    wandb.init(
        project="llm-training-pipeline",
        name=run_name,
        reinit="finish_previous",
        config={
            "model": model_id,
            "init_from": SFT_ADAPTER,
            "dataset": "CodeUltraFeedback_binarized",
            "min_rating_margin": MIN_RATING_MARGIN,
            "n_train": len(train_pairs),
            "beta": beta,
            "lora_rank": 16,
            "learning_rate": 5e-5,
            "epochs": epochs,
        },
    )

    trainer = DPOTrainer(
        model=policy,
        ref_model=None,                # None + peft_config => adapter-disabled reference
        args=args,
        train_dataset=train_pairs,
        eval_dataset=val_pairs,
        processing_class=tokenizer,
        peft_config=peft_config,
    )

    t0 = time.time()
    trainer.train()
    wall_min = (time.time() - t0) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    wandb.finish()

    history = trainer.state.log_history
    with open(f"results/dpo_{run_name}_log.json", "w") as f:
        json.dump(history, f, indent=2)

    def last(key):
        vals = [h[key] for h in history if key in h]
        return vals[-1] if vals else None

    metrics = {
        "run": run_name,
        "beta": beta,
        "n_train": len(train_pairs),
        "steps": total_steps,
        "final_train_loss": last("loss"),
        "eval_loss": last("eval_loss"),
        "eval_reward_accuracy": last("eval_rewards/accuracies"),
        "eval_reward_margin": last("eval_rewards/margins"),
        "eval_rewards_chosen": last("eval_rewards/chosen"),
        "eval_rewards_rejected": last("eval_rewards/rejected"),
        "peak_vram_gb": peak_gb,
        "wall_clock_min": wall_min,
    }
    # Implicit reward is beta * log(pi / pi_ref), so dividing it back out recovers the
    # mean log-ratio against the SFT reference -- a one-sided proxy for KL drift, not a
    # true KL, but it is what beta is directly trading against.
    if metrics["eval_rewards_chosen"] is not None:
        metrics["logratio_vs_sft"] = (
            metrics["eval_rewards_chosen"] + metrics["eval_rewards_rejected"]
        ) / (2 * beta)

    if save_adapter:
        trainer.save_model(f"checkpoints/{run_name}/final")
        tokenizer.save_pretrained(f"checkpoints/{run_name}/final")
        print(f"adapter saved to checkpoints/{run_name}/final")

    del trainer, policy
    gc.collect()
    torch.cuda.empty_cache()

    print(json.dumps(metrics, indent=2))
    return metrics


DPO_RUNS = {}


In [ ]:

# beta=0.1: the standard setting from the DPO paper. This is the checkpoint that carries
# forward into Phase 4, so it is the one that gets saved.
DPO_RUNS["dpo-beta0.1"] = run_dpo("dpo-beta0.1", beta=0.1, save_adapter=True)


In [ ]:

# beta=0.5: a 5x tighter KL leash. Expect smaller reward margins and a smaller log-ratio
# against SFT -- the policy is allowed to move less, so it separates chosen from rejected
# less. If reward accuracy holds up anyway, the preference signal was easy to fit; if it
# collapses, beta was the binding constraint.
DPO_RUNS["dpo-beta0.5"] = run_dpo("dpo-beta0.5", beta=0.5)


In [ ]:

runs = list(DPO_RUNS.values())


def cell(v, spec="{:.4f}"):
    return "-" if v is None else spec.format(v)


lines = [
    "# DPO beta ablation",
    "",
    f"Policy initialized from the Phase 2 SFT checkpoint (`{SFT_ADAPTER}`, LoRA r={SFT_LORA_RANK}) "
    f"merged into `{model_id}` and re-quantized to 4-bit NF4.",
    f"Preference data: `coseal/CodeUltraFeedback_binarized`, filtered to rating margin "
    f">= {MIN_RATING_MARGIN}, {len(train_pairs)} train / {len(val_pairs)} val pairs.",
    "DPO adapters are LoRA r=16, lr 5e-5 cosine, 1 epoch, batch 1 x grad_accum 16, seed 42. "
    "Only `beta` varies.",
    "",
    "| Run | beta | Train loss | Eval loss | Reward accuracy | Reward margin | log(pi/pi_sft) | Peak VRAM (GB) | Wall clock (min) |",
    "|---|---|---|---|---|---|---|---|---|",
]
for r in runs:
    lines.append(
        f"| {r['run']} | {r['beta']} | {cell(r['final_train_loss'])} | {cell(r['eval_loss'])} | "
        f"{cell(r['eval_reward_accuracy'])} | {cell(r['eval_reward_margin'])} | "
        f"{cell(r.get('logratio_vs_sft'))} | {cell(r['peak_vram_gb'], '{:.2f}')} | "
        f"{cell(r['wall_clock_min'], '{:.1f}')} |"
    )

lines += [
    "",
    "Reward accuracy is the fraction of held-out pairs where the policy assigns the chosen "
    "response a higher implicit reward than the rejected one. 0.5 is chance; anything near "
    "chance means DPO did not learn the preference, not that the preference was subtle.",
]

with open("results/dpo_ablations.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")

print("\n".join(lines))
print("\nwrote results/dpo_ablations.md")


In [ ]:

# Qualitative SFT vs DPO on the *same* 20 prompts used in Phase 2, so all three stages
# (base / SFT / DPO) are answering an identical held-out set.
with open("results/sft_qualitative.json", encoding="utf-8") as f:
    PROMPTS = [row["prompt"] for row in json.load(f)]
print(f"loaded {len(PROMPTS)} prompts from the SFT comparison")

dpo_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
dpo_model = PeftModel.from_pretrained(dpo_model, "checkpoints/dpo-beta0.1/final")
dpo_model.eval()
dpo_model.config.use_cache = True


@torch.no_grad()
def generate(instruction, max_new_tokens=200, adapter=True):
    """Greedy decode. adapter=False disables the DPO LoRA, which *is* the SFT model --
    same weights, same quantization, so the comparison isolates DPO exactly."""
    inputs = tokenizer(
        PROMPT_TEMPLATE.format(instruction=instruction), return_tensors="pt"
    ).to(dpo_model.device)
    kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    if adapter:
        out = dpo_model.generate(**inputs, **kwargs)
    else:
        with dpo_model.disable_adapter():
            out = dpo_model.generate(**inputs, **kwargs)
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )


rows = []
for i, prompt in enumerate(PROMPTS, 1):
    print(f"[{i:>2}/{len(PROMPTS)}] {prompt.splitlines()[0][:60]}")
    rows.append({
        "prompt": prompt,
        "sft": generate(prompt, adapter=False),
        "dpo": generate(prompt, adapter=True),
    })

with open("results/dpo_qualitative.json", "w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2)


In [ ]:

# Length inflation is the canonical DPO reward-hacking symptom: preference data tends to
# rate longer answers higher, so the policy learns verbosity as a proxy for quality.
# Measure it rather than eyeballing it.
sft_chars = sum(len(r["sft"]) for r in rows) / len(rows)
dpo_chars = sum(len(r["dpo"]) for r in rows) / len(rows)
sft_toks = sum(n_tokens(r["sft"]) for r in rows) / len(rows)
dpo_toks = sum(n_tokens(r["dpo"]) for r in rows) / len(rows)
identical = sum(r["sft"].strip() == r["dpo"].strip() for r in rows)

summary = [
    "# SFT vs DPO: 20 fixed prompts",
    "",
    f"- **SFT:** `{model_id}` + Phase 2 LoRA r={SFT_LORA_RANK}, merged (DPO adapter disabled)",
    "- **DPO:** same weights + DPO LoRA r=16, beta=0.1, CodeUltraFeedback pairs",
    "- **Decoding:** greedy (`do_sample=False`), `max_new_tokens=200`, identical template",
    "",
    "| Metric | SFT | DPO |",
    "|---|---|---|",
    f"| Mean response length (chars) | {sft_chars:.0f} | {dpo_chars:.0f} |",
    f"| Mean response length (tokens) | {sft_toks:.0f} | {dpo_toks:.0f} |",
    f"| Byte-identical to SFT | - | {identical}/{len(rows)} |",
    "",
    f"Length ratio DPO/SFT: **{dpo_chars / sft_chars:.2f}x**. A ratio well above 1.0 with no "
    "matching quality gain is length bias, not improvement.",
    "",
]
for i, r in enumerate(rows, 1):
    summary += [
        "---", "", f"## {i}. {r['prompt']}", "",
        "**SFT**", "", "```", r["sft"], "```", "",
        "**DPO**", "", "```", r["dpo"], "```", "",
    ]

with open("results/dpo_qualitative.md", "w", encoding="utf-8") as f:
    f.write("\n".join(summary) + "\n")

print("\n".join(summary[:20]))
print("\nwrote results/dpo_qualitative.md")


In [ ]:

# results/ has never actually been committed -- the README claims it is. Stage it here so
# the metrics live in the repo rather than only in Drive and wandb.
!git add results/ 03_dpo.ipynb
!git status --short
